## Synthetic data generator — subscription pricing experiments

This notebook documents and runs the **vectorized simulator** documented in `README.md`.

**Structural flow:** users → latent factors → RCT assignment → funnel (view → click) → logistic conversion → geometric retention → skewed revenue/LTV → deliberate messiness → CSV export.

**Important:** latent willingness to pay and related constructs are simulated but **stripped before export**.


In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.generator import generate, SimulationConfig
from src import simulation as sim

SEED = 42
N_USERS = 10_000  # Set to 1_000_000 for large-scale (ensure RAM)
OUTPUT_DIR = ROOT / "output"

print("ROOT", ROOT)


### Structural equations (reference)

**Conversion** uses a logistic link with assigned price, observed engagement, student/trial indicators, and **latent** terms (WTP gap, price sensitivity × price, intrinsic engagement, brand affinity). See `src/simulation.py` functions `build_conversion_logit` and `assign_treatment`.

**Churn** uses a **geometric** survival model: each month after the first payment, churn with probability $q_i$ driven by price vs latent WTP, observed engagement, billing failures, and latent traits (`build_churn_probability`).

**Revenue** is built from months active × paid price, minus refunds and promotional credits, plus noise; LTV applies a simple discrete discount factor.


In [ ]:
result = generate(n_users=N_USERS, seed=SEED, output_dir=OUTPUT_DIR, write_summary=True)
users = result["tables"]["users"]
experiments = result["tables"]["pricing_experiments"]
subs = result["tables"]["subscriptions"]
print(json.dumps(result["summary"], indent=2))


In [ ]:
# Sanity: latent columns must not appear in exports
for name, df in {"users": users, "pricing_experiments": experiments, "subscriptions": subs}.items():
    bad = {"willingness_to_pay","price_sensitivity","intrinsic_engagement","latent_brand_affinity"} & set(df.columns)
    assert not bad, (name, bad)
print(users.shape, experiments.shape, subs.shape)
display(users.head(3))
display(experiments.head(3))
display(subs.head(3))


### Faker: vendor-style auxiliary ids (demo only)

The core pipeline avoids per-row Python loops for scalability. Below we use **Faker** purely as a pedagogical companion to illustrate realistic opaque identifiers for a tiny sample (`n_demo`). Production-scale runs should keep vectorized ids as in `users.csv`.


In [ ]:
from faker import Faker

fake = Faker()
fake.seed_instance(SEED)
n_demo = min(250, len(users))
vendor_tokens = pd.DataFrame({
    "user_id": users["user_id"].iloc[:n_demo].values,
    "vendor_audience_token": [fake.hexify('^^^^^^^^^^^^^^^^')[:8] for _ in range(n_demo)],
})
vendor_tokens.to_csv(OUTPUT_DIR / "vendor_audience_token_sample.csv", index=False)
vendor_tokens.head()


### Scaling notes

- All heavy draws use **NumPy vectorization** (`np.random.Generator`).
- Disk output is CSV; for millions of rows consider **Parquet** via `df.to_parquet`.
- Duplicate `pricing_experiments` rows intentionally break strict one-row-per-user grain — dedupe on `user_id` with `experiment_row_id.min()` for person-level metrics.
